# 📈 Notebook 03 — Running Aggregates & Rolling Windows

**Functions covered:** `SUM() OVER`, `AVG() OVER`, `COUNT() OVER`, `MIN/MAX() OVER`, rolling frames, cumulative metrics

## Frame Syntax

```sql
SUM(amount) OVER (
    PARTITION BY region
    ORDER BY sale_date
    ROWS BETWEEN 2 PRECEDING AND CURRENT ROW  -- 3-row rolling window
)
```

Common frames:
| Frame | Meaning |
|-------|---------|
| `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` | Cumulative (running total) |
| `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` | Rolling 3-row window |
| `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING` | Entire partition (same as no frame) |
| `RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND CURRENT ROW` | 30-day rolling window (date-based) |


In [ ]:
import duckdb, pandas as pd

employees = pd.read_csv("../data/employees.csv")
sales     = pd.read_csv("../data/sales.csv")
orders    = pd.read_csv("../data/orders.csv")
con = duckdb.connect()
con.register("employees", employees)
con.register("sales",     sales)
con.register("orders",    orders)

def sql(query):
    return con.execute(query).fetchdf()


---
## Exercise 1 · Running Total of Daily Sales

**Question:** Calculate the **cumulative (running) total of sales amount** across all reps, ordered by date. Show `sale_date`, `daily_total`, `running_total`. One row per day.

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
SELECT
    sale_date,
    daily_total,
    SUM(daily_total) OVER (
        ORDER BY sale_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total
FROM (
    SELECT sale_date, SUM(amount) AS daily_total
    FROM sales
    GROUP BY sale_date
)
ORDER BY sale_date;
```
</details>

---
## Exercise 2 · 3-Month Rolling Average

**Question:** For each month, compute the **3-month rolling average of total sales** (current month + 2 prior months). Show `month_label`, `monthly_total`, `rolling_3mo_avg`. Round to 2 decimals.

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
SELECT
    month_label,
    monthly_total,
    ROUND(
        AVG(monthly_total) OVER (
            ORDER BY month_label
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2
    ) AS rolling_3mo_avg
FROM (
    SELECT month_label, SUM(amount) AS monthly_total
    FROM sales
    GROUP BY month_label
)
ORDER BY month_label;
```
</details>

---
## Exercise 3 · Each Sale's % of Company Total

**Question:** Show each individual sale's `amount` as a **percentage of total company revenue** (all time). Return `rep_name`, `sale_date`, `amount`, `pct_of_total` (rounded to 4 decimal places).

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
SELECT
    rep_name,
    sale_date,
    amount,
    ROUND(
        amount / SUM(amount) OVER () * 100, 4
    ) AS pct_of_total
FROM sales
ORDER BY pct_of_total DESC;
```
</details>

---
## Exercise 4 · Each Sale's % of Regional Total

**Question:** For each sale, compute its **percentage of the total revenue for that region** (all time). Show `region`, `rep_name`, `amount`, `regional_total`, `pct_of_region`. Order by region, then pct desc.

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
SELECT
    region,
    rep_name,
    amount,
    SUM(amount) OVER (PARTITION BY region) AS regional_total,
    ROUND(
        amount / SUM(amount) OVER (PARTITION BY region) * 100, 2
    ) AS pct_of_region
FROM sales
ORDER BY region, pct_of_region DESC;
```
</details>

---
## Exercise 5 · Cumulative Sales Milestones

**Question:** For each sales rep, show their **running cumulative sales count and amount** (chronologically). Identify the row where they crossed the **$500,000 cumulative threshold** for the first time. Show `rep_name`, `sale_date`, `amount`, `cum_count`, `cum_amount`.

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
-- Running cumulative per rep
SELECT
    rep_name,
    sale_date,
    amount,
    COUNT(*) OVER (
        PARTITION BY rep_name
        ORDER BY sale_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cum_count,
    SUM(amount) OVER (
        PARTITION BY rep_name
        ORDER BY sale_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cum_amount
FROM sales
ORDER BY rep_name, sale_date;

-- Bonus: first row crossing $500k
-- SELECT * FROM (...above as cte...)
-- WHERE cum_amount >= 500000
-- QUALIFY ROW_NUMBER() OVER (PARTITION BY rep_name ORDER BY sale_date) = 1;
```
</details>

---
## Exercise 6 · YTD Sales by Region

**Question:** For each month in 2023, calculate the **year-to-date (YTD) cumulative total by region**. Show `region`, `month_label`, `monthly_total`, `ytd_total`. Only 2023 data.

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
SELECT
    region,
    month_label,
    monthly_total,
    SUM(monthly_total) OVER (
        PARTITION BY region
        ORDER BY month_label
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS ytd_total
FROM (
    SELECT region, month_label, SUM(amount) AS monthly_total
    FROM sales
    WHERE year = 2023
    GROUP BY region, month_label
)
ORDER BY region, month_label;
```
</details>

---
## Exercise 7 · Salary vs. Department Average

**Question:** For each employee, show their salary alongside the **department average salary** and compute how much they earn **above or below the average** (as both an absolute dollar amount and a percentage). Show `full_name`, `department`, `salary`, `dept_avg_salary`, `diff_from_avg`, `pct_above_avg`.

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
SELECT
    full_name,
    department,
    salary,
    ROUND(AVG(salary) OVER (PARTITION BY department), 0) AS dept_avg_salary,
    salary - ROUND(AVG(salary) OVER (PARTITION BY department), 0) AS diff_from_avg,
    ROUND(
        (salary - AVG(salary) OVER (PARTITION BY department))
        / AVG(salary) OVER (PARTITION BY department) * 100, 1
    ) AS pct_above_avg
FROM employees
ORDER BY department, salary DESC;
```
</details>

---
## Exercise 8 · Rolling Max Sale per Rep

**Question:** For each rep's sales (in chronological order), show the **running maximum sale amount** they have achieved so far at each point in time. This represents their "personal best" up to that date. Show `rep_name`, `sale_date`, `amount`, `running_best`.

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
SELECT
    rep_name,
    sale_date,
    amount,
    MAX(amount) OVER (
        PARTITION BY rep_name
        ORDER BY sale_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_best
FROM sales
ORDER BY rep_name, sale_date;
```
</details>

---
## 🎯 Summary

- `SUM/AVG/COUNT/MIN/MAX() OVER (ORDER BY ... ROWS BETWEEN ...)` → all aggregate functions work as window functions.
- **No frame clause** → the entire partition is the window (useful for % of total).
- **`ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`** → running/cumulative total.
- **`ROWS BETWEEN n PRECEDING AND CURRENT ROW`** → rolling window of n+1 rows.
- Use `PARTITION BY` to reset the running total per group.

➡️ **Next:** `04_advanced_challenges.ipynb` — real-world data-engineer scenarios